<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 08: Proyecto Capstone</p>
  </div>
</div>

# Proyecto Final · Plantilla CRISP-DM

Esta es la **plantilla** de trabajo del proyecto final. Clónenla y complétenla con el caso seleccionado por su equipo:

1. Elijan **un** problema: **regresión**, **clasificación** o **pronóstico de series temporales** (*forecasting*).
2. Elijan un **dataset del catálogo curado** (o datos propios de su organización, **sin datos sensibles ni confidenciales**).
3. Recorran las **fases de CRISP-DM** de abajo, reemplazando las directrices y los bloques `# TODO` con su desarrollo.

**Reglas** (ver la pauta del Proyecto Final): fijen la **semilla** (`SESSION_ID`), carguen los datos por **URL pública estable**, realicen **todo el modelado con PyCaret o StatsForecast**, y **declaren el uso de IA** en el anexo correspondiente. El notebook debe ejecutarse completo con *Ejecutar todo* y reproducir **exactamente los mismos resultados**.

## Preparación del entorno

Instalación de las librerías del módulo. Utilizarán **PyCaret** (problemas tabulares) **o** **StatsForecast** (series temporales), según el enfoque de su proyecto.

In [ ]:
%%capture
!pip install -q pycaret==4.0.0a8 "statsforecast==2.0.3" "utilsforecast==0.2.15" \
    fg-data-profiling missingno plotly ipywidgets shap "setuptools<81"
# Opcional (más modelos para compare_models): !pip install -q xgboost catboost lightgbm optuna

## Setup técnico

Importaciones y configuración general (no es contenido del proyecto).

In [ ]:
%matplotlib inline
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px

SESSION_ID = 42   # Semilla fija: el trabajo debe ser REPLICABLE

## Selección del problema y carga de datos

- **Tipo de problema**: regresión / clasificación / pronóstico de series temporales (*forecasting*).
- **Dataset**: del catálogo de la pauta (o propio, resguardando la confidencialidad).

**Las celdas de carga de cada dataset del catálogo se encuentran en el notebook `datasets`** (una por dataset, con columnas en español y tipos corregidos). Ábranlo, **copien la celda correspondiente** y péguenla en el bloque siguiente:

[![Abrir el notebook de datasets en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastiancontz/ust-fundamentos-machine-learning-colab/blob/main/ediciones/2026/notebooks/datasets.ipynb)

Si utilizan **datos propios**, cárguenlos desde un repositorio accesible (ajustando al formato del archivo):

```python
URL = "https://.../dataset_propio.csv"
df = pd.read_csv(URL)     # Ajustar separador o codificación si corresponde
df.head()
```

In [ ]:
# Pegar aquí la celda de carga del dataset (desde el notebook datasets), o cargar los datos propios.


## Fase 1 · Comprender el negocio

Antes de entrenar un modelo, definan en texto:

- **Pregunta de negocio**: ¿qué **decisión concreta** ayuda a fundamentar el modelo?
- **Métrica ligada a la decisión**: aquella que responde a la pregunta estratégica del negocio (distinguiendo el éxito técnico del impacto organizacional).
- **Línea base** (*baseline*): la vara mínima a superar — regla mayoritaria o promedio (tabular), o *Naive* / *SeasonalNaive* (series). *Un modelo que no supera la línea base no aporta valor.*

> _Escriban aquí la pregunta de negocio de su equipo, la métrica principal y la línea base._

## Fase 2 · Comprender los datos (EDA y valores atípicos)

Inspeccionen los datos antes de modelar: tipos de variables, distribuciones, correlaciones y calidad. Detecten y **decidan** el tratamiento para valores faltantes y atípicos (*outliers*), documentando el criterio aplicado.

```python
df.info(); df.describe()

# Reporte automático de análisis exploratorio (como en la Clase 2)
from data_profiling import ProfileReport
ProfileReport(df).to_notebook_iframe()

# Inspección de valores atípicos (boxplot) y distribuciones
px.box(df, y="variable_numerica")
px.histogram(df, x="variable_interes")
```

In [ ]:
# TODO: análisis exploratorio (EDA) y tratamiento de valores atípicos


## Fase 3 · Preparar los datos (sin fuga)

**Resguarden la honestidad analítica**: el preprocesamiento no debe utilizar información de la muestra de prueba (*test*), y ninguna variable debe incorporar información del resultado futuro.

**Para problemas TABULARES** — utilicen el pipeline de PyCaret (partición train/test, imputación y escalado):

```python
from pycaret.tasks import ClassificationExperiment   # o RegressionExperiment
exp = ClassificationExperiment(
    target="nombre_del_target", session_id=SESSION_ID, train_size=0.7,
    fold=5, normalize=True, fold_strategy="stratifiedkfold",   # Estratificada en clasificación
).fit(df)
```

**Para problemas de PRONÓSTICO (series temporales)** — estructuren la serie en formato `unique_id` / `ds` / `y`:

```python
serie = df.rename(columns={"columna_fecha": "ds", "columna_valor": "y"})
serie["unique_id"] = "serie_1"
serie = serie[["unique_id", "ds", "y"]].sort_values("ds")
```

In [ ]:
# TODO: preparación de datos (setup de PyCaret o formato unique_id/ds/y)


## Fase 4 · Modelar y comparar contra la línea base

Comparen múltiples modelos, evalúen según las **métricas relevantes** y afinen hiperparámetros donde corresponda.

**TABULAR (PyCaret):**

```python
mejores = exp.compare_models(sort="AUC")          # En regresión: sort="RMSE"
base = exp.create_model("rf")                      # Modelo base seleccionado; vive en base.pipeline
afinado = exp.tune_model(base.pipeline, n_iter=30, optimize="AUC")   # Comparar base vs. afinado en holdout
```

**PRONÓSTICO DE SERIES (StatsForecast):**

```python
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA
sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=12), AutoETS(season_length=12), AutoARIMA(season_length=12)],
    freq="MS", n_jobs=-1)     # Ajustar season_length y frecuencia según la serie
```

In [ ]:
# TODO: entrenamiento y comparación de modelos frente a la línea base


## Fase 4b · Segmentación con clustering — **solo TABULAR**

Incorporen una etapa de segmentación no supervisada (como en la Clase 5). **En series de tiempo este paso se omite**: se sustituye por la descomposición y lectura de la anatomía de la serie (tendencia, estacionalidad y quiebres).

```python
from pycaret.tasks import ClusteringExperiment
cexp = ClusteringExperiment(session_id=SESSION_ID, normalize=True).fit(df)
km = cexp.create_model("kmeans", n_clusters=4)     # Atención: parámetro n_clusters
etiquetado = cexp.assign_model(km.pipeline)         # Perfilar segmentos y asignar un nombre accionable
```

In [ ]:
# TODO (solo tabular): segmentación (clustering) y perfilado de grupos


## Fase 5 · Evaluar e interpretar

Evaluación **honesta** (muestra de prueba *holdout* intacta o validación temporal con ventanas móviles) e **interpretación** orientada a fundamentar la decisión.

**TABULAR:**

```python
exp.predict_model(base.pipeline).metrics             # Métricas en el holdout (una única evaluación final)
exp.plot_model(base.pipeline, plot="shap_beeswarm")  # Qué variables influyen en la predicción
exp.plot_model(base.pipeline, plot="pdp", feature="variable_focal")   # Gráfico de dependencia parcial
```

> **Consideración técnica (como en la Clase 6):** para que `shap_beeswarm` y `pdp` operen sin conflictos, el modelo requiere **variables numéricas** y el **target binario en 0/1** (con variables categóricas codificadas como `float`).

**PRONÓSTICO DE SERIES (validación temporal + intervalos):**

```python
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, mape
H = 12
cv = sf.cross_validation(df=serie, h=H, n_windows=3, step_size=12)
evaluate(cv.drop(columns="cutoff"), metrics=[mae, mape],
         models=["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA"])
fc = sf.forecast(df=serie, h=H, level=[80, 95])       # Pronóstico con intervalos de predicción
```

In [ ]:
# TODO: evaluación sin fuga e interpretabilidad conectada a la decisión


## Fase 5b · Del modelo al valor (costos y beneficios)

Traduzcan las métricas técnicas a **impacto económico en pesos**: asignen un costo y beneficio a cada acierto y error en la matriz de decisión y compárenlo contra la línea base o el escenario sin intervención. La métrica orienta; el **costo decide**.

> _Escriban aquí la estimación de valor esperado para su organización (tabla de costos y beneficios)._

In [ ]:
# TODO: marco de valor esperado y balance económico


## Fase 6 · Comunicar y guardar

**Persistencia del modelo** para su aplicación en nuevos registros (problemas tabulares):

```python
final = exp.finalize_model(base.pipeline)
exp.save_model(final.pipeline, "modelo_final")   # Recuperable con load_model
```

**Estructura del informe ejecutivo** (≤ 5 páginas, dirigido a dirección no técnica): problema de negocio → hallazgos exploratorios → comparación contra línea base → impacto económico → recomendación y límites. **Prioricen gráficos antes que viñetas de texto.**

In [ ]:
# TODO (tabular): persistencia con finalize_model y save_model


## Anexo · Declaración de uso de IA

El uso de herramientas de IA generativa está autorizado, pero **debe declararse con transparencia** (su omisión conlleva penalización en la rúbrica). Completen la siguiente tabla detallando cada interacción:

| Etapa o componente | Herramienta o modelo | Prompt o instrucción aplicada (resumen) |
|---|---|---|
|  |  |  |

## Lista de verificación de entrega

- [ ] Ejecución completa con *Ejecutar todo*, semillas fijas y carga de datos por URL pública.
- [ ] Modelado implementado exclusivamente con PyCaret o StatsForecast.
- [ ] EDA + valores atípicos · comparación contra línea base · métricas interpretadas.
- [ ] Modelo optimizado mediante ajuste de hiperparámetros (donde aplique).
- [ ] Segmentación con clustering (tabular) o descomposición estacional (series temporales).
- [ ] Interpretabilidad (valores SHAP / dependencia o componentes / intervalos) vinculada a una acción.
- [ ] Marco de valor esperado (cuantificación en costos y beneficios).
- [ ] Persistencia del modelo final con `save_model` (tabular).
- [ ] Informe ejecutivo ≤ 5 páginas + anexo de declaración de uso de IA.

## Atribución de datos

**Creador:** CNE, MINSAL-DEIS, Department for Transport (GB), datos.gob.cl y Department of Transport and Planning (Victoria).
**Fuente:** [catálogo de datos abiertos de Chile](https://datos.gob.cl/), [data.gov.uk](https://www.data.gov.uk/) y [datos abiertos de Victoria](https://discover.data.vic.gov.au/).
**Licencia:** combinación de [CC0](https://creativecommons.org/publicdomain/zero/1.0/), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), [OGL v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/) y [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/), según el dataset elegido.
**Modificación:** columnas traducidas, tipos corregidos, recorte de campos y agregaciones temporales documentadas en el catálogo del proyecto.